# 1、SummarizationMiddleware中间件

## 举例1：测试trigger、keep参数

In [1]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

In [9]:

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.agents import create_agent

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]

agent = create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.001)
            ],
            keep=("messages", 2)
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user is having a simple friendly introduction/conversation in Chinese and wants to identify the assistant.

## SUMMARY
The conversation is a casual greeting exchange:
- The user said hello and asked who the assistant is.
- The assistant replied: “你好老王，我是小王” (“Hello Lao Wang, I’m Xiao Wang”).
- The user responded positively: “好的小王，很高兴认识你” (“Okay Xiao Wang, nice to meet you”).

No decisions, tasks, or follow-up requests were made.

## ARTIFACTS
None

## NEXT STEPS
None required unless the user continues the conversation.
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哈哈，开个玩笑啦。我的意思不是“扫兴”，而是想说：很高兴认识你，后面你想聊什么都可以。  
如果你愿意，我可以继续当你的

In [11]:

from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.agents import create_agent


messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]



agent = create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages",2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

历史消息摘要：
老王与助手进行了友好的自我介绍。老王先问“你是谁”，助手回答自己是“小王”。随后老王表示很高兴认识“小王”。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

开个玩笑啦，别当真🙂  
我意思是：刚刚那句“你高兴得太早了”有点调侃，像是在故意接一句不按常理的话。其实我也是很高兴认识你的。

如果你愿意，我们可以继续轻松聊聊。
